<a href="https://colab.research.google.com/github/Raijeku/quantum-eigengame/blob/main/EigenGame_theory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [1]:
import autograd.numpy as np
import matplotlib.pyplot as plt
#import pennylane as qml
#from pennylane import qaoa
#from pennylane import numpy as np
from scipy.linalg import expm, sqrtm, eigh
import networkx as nx
from jax import grad, jacfwd
from jax.lax import stop_gradient
import jax.numpy as jnp
import numpy

# Functions

In [3]:
from jax import vjp

def generate_matrix(eigenvalues):
  D = np.diag(eigenvalues)
  P = np.random.rand(D.shape[0], D.shape[0])
  norm = np.linalg.norm(P)
  P = P/norm
  np.fill_diagonal(P, 1)
  return np.dot(P, np.dot(D, np.linalg.inv(P))), P

# Variational functions
def R(theta):
  return jnp.array([[jnp.cos(theta), -jnp.sin(theta)],
                    [jnp.sin(theta), jnp.cos(theta)]])

def U3(theta_3):
  return jnp.array([[jnp.cos(theta_3[0]/2), -jnp.exp(1j*theta_3[2])*jnp.sin(theta_3[0]/2)],
                   [jnp.exp(1j*theta_3[1])*jnp.sin(theta_3[0]/2), jnp.exp(1j*(theta_3[1]+theta_3[2]))*jnp.cos(theta_3[0]/2)]])

def ansatz(theta, gate='R'):
  qubits, layers = theta.shape[:2]
  n = 2**qubits

  if gate=='R':
    U_s = R
  elif gate=='U3':
    U_s = U3

  U = jnp.eye(n)
  for l in range(layers):
    U_l = U_s(theta[0,l])
    for k in range(1, qubits):
      U_l = jnp.kron(U_l, U_s(theta[k, l]))
    U = jnp.matmul(U, U_l)

  return U

def v(theta_index, gate='R'):
  qubits = theta_index.shape[0]
  n = 2**qubits

  s = jnp.ones(n)
  s /= jnp.linalg.norm(s)

  phi = jnp.matmul(ansatz(theta_index, gate), s)

  return phi

def d_v(theta_index, gate='R'):
  return jacfwd(v)(theta_index, gate)

def rewards_var(theta_i, X, gate='R'):
  return jnp.linalg.norm(X @ v(theta_i, gate))**2

def penalties_var(theta_i, X, thetas=[], gate='R'):
  return jnp.array([(jnp.conj(X @ v(theta_i, gate)) @ X @ v(theta_j, gate))**2 / (jnp.linalg.norm(X @ v(theta_j, gate)))**2 for theta_j in thetas]).sum(axis=0)

def agent_utility_var(theta_i, X, thetas=[], gate='R'):
  return rewards_var(theta_i, X, gate) - penalties_var(theta_i, X, thetas, gate)

def d_rewards_var(theta_i, X, gate='R'):
  return grad(rewards_var)(theta_i, X, gate)

def d_penalties_var(theta_i, X, thetas=[], gate='R'):
  if len(thetas) == 0:
    return grad(penalties_var)(theta_i, X, thetas, gate)
  else:
    return grad(penalties_var, holomorphic = True)(theta_i, X, thetas, gate)

def d_agent_utility_var(theta_i, X, thetas=[], gate='R'):
  return d_rewards_var(theta_i, X, gate) - d_penalties_var(theta_i, X, thetas, gate)


# Variational tests

qubits = 3
layers = 2
n = 2**qubits

# Using U3 gates
theta_init = np.random.rand(qubits, layers)

eigs = np.random.rand(n)
X, _ = generate_matrix(eigs)

print('Exact gradient:')
print(d_agent_utility_var(theta_init, X), d_agent_utility_var(theta_init, X).shape)

# Normal functions

def rewards(v_i, X):
  return jnp.linalg.norm(X @ v_i)**2

def penalties(v_i, X, v=[]):
  return jnp.array([(jnp.conj(X @ v_i) @ X @ v_j)**2 / (jnp.linalg.norm(X @ v_i))**2 for v_j in v]).sum(axis=0)

def agent_utility(v_i, X, v=[]):
  return rewards(v_i, X) - penalties(v_i, X, v)

def d_rewards(v_i, X):
  return grad(rewards)(v_i, X)

def d_penalties(v_i, X, v=[]):
  return grad(penalties)(v_i, X, v)
  #if len(v) == 0:
  #  return grad(penalties)(v_i, X, v)
  #else:
  #  return grad(penalties, holomorphic = True)(v_i, X, v)

def d_agent_utility(v_i, X, v=[]):
  return d_rewards(v_i, X) - d_penalties(v_i, X, v)

def d_agent_utility_R(v_i, X, v=[]):
  grad_u = d_rewards(v_i, X) - d_penalties(v_i, X, v)

  return grad_u - np.dot(grad_u.T, v_i) * v_i

# Normal tests
n = 2**qubits

# Using U3 gates
v_init = np.random.rand(n)

eigs = np.random.rand(n)
X, _ = generate_matrix(eigs)

print('Parameterized gradient:')
print(d_agent_utility(v_init, X), d_agent_utility(v_init, X).shape)

def d_agent_utility_epsilon(v_i, X, v = [], epsilon = 10e-5):
  d = X.shape[0]
  gradient = []
  for m in range(d):
    epsilon_vector = np.zeros(d)
    epsilon_vector[m] = epsilon
    v_i_epsilon = v_i + epsilon_vector
    term_0 = agent_utility(v_i_epsilon, X, v)
    term_1 = agent_utility(v_i, X, v)
    term = (term_0 - term_1) / epsilon
    gradient.append(term)
  return np.array(gradient)

epsilon = 10e-7

print('Finite differences gradient:')
print(d_agent_utility_epsilon(v_init, X, [], epsilon), d_agent_utility_epsilon(v_init, X, [], epsilon).shape)

Exact gradient:
[[-0.05377889 -0.05377889]
 [-0.13374075 -0.13374074]
 [-0.2006876  -0.2006876 ]] (3, 2)
Parameterized gradient:
[ 0.09043073  0.06025548  0.44927964  0.11081583 -0.01778281  0.09814858
  1.0474324  -0.02971929] (8,)
Finite differences gradient:
[ 0.08940697  0.          0.4172325   0.08940697 -0.08940697  0.08940697
  1.013279   -0.08940697] (8,)


In [4]:
def randomize_angle(ref_angle):
  sat_angle = np.random.uniform(low=0, high=ref_angle)

  return sat_angle

def initialize_relative_vec(ref_angle, ref_vec):
  a = np.random.rand(n)
  a -= a.dot(eigvecs[0]) * ref_vec
  a /= np.linalg.norm(a)

  v = eigvecs[0]*np.cos(ref_angle) + a*np.sin(ref_angle)

  return v

In [ ]:
eigs = np.random.rand(n)
X, _ = generate_matrix(eigs)
M = X.T @ X
M /= np.linalg.norm(M)
k = M.shape[0]

eigs, eigvecs = np.linalg.eig(M)
eigs, eigvecs = zip(*sorted(zip(eigs, eigvecs), reverse=True))
print('Eigenvalues:', eigs)
print('Eigenvectors:', eigvecs)

Eigenvalues: (0.6084751238316147, 0.5346559827316114, 0.3919390120772832, 0.32740530318891803, 0.22370219577798023, 0.14713858021708304, 0.10450206047251497, 0.02185102367996885)
Eigenvectors: (array([ 0.06560434, -0.00159593,  0.0410363 , -0.12090459, -0.05726468,
       -0.9875474 ,  0.021216  ,  0.02030735]), array([-5.70149492e-01,  8.06871171e-01,  1.41839303e-01,  2.68446166e-02,
       -4.28691660e-02, -3.42148586e-02,  1.60963227e-04, -6.40320452e-03]), array([ 0.09516309,  0.01162895,  0.03150465,  0.020384  , -0.98707824,
        0.06146303, -0.0921671 ,  0.05296008]), array([-0.15355929,  0.06264969, -0.9761901 , -0.12037676, -0.05305969,
       -0.03321005,  0.01921386, -0.02773931]), array([ 0.06243428,  0.0371388 ,  0.01202138,  0.03465869, -0.08443547,
        0.02620404,  0.99259939, -0.01753253]), array([-0.02302874, -0.03335131,  0.03487852, -0.01579776, -0.05451175,
       -0.01583418, -0.01899751, -0.99664845]), array([-0.79521553, -0.58362926,  0.09887569, -0.04305

# Chain rule gradient verification

In [9]:
qubits = 3
layers = 2
n = 2**qubits

# Using U3 gates
theta_init = np.random.rand(qubits, layers)
v_init = v(theta_init, gate='R')

eigs = np.random.rand(n)
X, _ = generate_matrix(eigs)

print('df/dtheta:')
print(d_agent_utility_var(theta_init, X, gate='R'), d_agent_utility_var(theta_init, X, gate='R').shape)

print('df/dv:')
print(d_agent_utility(v_init, X), d_agent_utility(v_init, X).shape)

print('dv/dtheta:')
print(d_v(theta_init).T, d_v(theta_init).T.shape)

print(f'||dv/dtheta||: {np.linalg.norm(d_v(theta_init).T)} = sqrt({qubits}*{layers})')

print('df/dv^T * dv/dtheta:')
print(np.squeeze(np.dot(np.conjugate(d_v(theta_init, gate='R')).T, d_agent_utility(v_init, X)[:, np.newaxis])).T, np.squeeze(np.dot(np.conjugate(d_v(theta_init, gate='R')).T, np.conjugate(d_agent_utility(v_init, X))[:, np.newaxis])).T.shape)



df/dtheta:
[[-0.1321987  -0.13219872]
 [-0.17382762 -0.17382756]
 [ 0.7992858   0.79928577]] (3, 2)
df/dv:
[ 0.14682531 -0.10638711  0.12319519 -0.00702197  0.01764743 -0.12835336
 -1.0857948   0.15272817] (8,)
dv/dtheta:
[[[-0.18026519  0.34936523  0.41168392 -0.79786927 -0.03649323
    0.07072618  0.08334206 -0.16152216]
  [-0.08334206  0.16152216 -0.03649323  0.07072618  0.41168392
   -0.79786927  0.18026519 -0.34936523]
  [-0.07072618 -0.03649323  0.16152216  0.08334206  0.34936523
    0.18026519 -0.79786927 -0.41168392]]

 [[-0.18026519  0.34936526  0.41168395 -0.79786927 -0.03649325
    0.07072621  0.08334206 -0.16152216]
  [-0.08334209  0.16152221 -0.03649323  0.0707262   0.4116839
   -0.79786927  0.18026516 -0.34936526]
  [-0.07072621 -0.03649325  0.16152222  0.08334212  0.34936526
    0.18026517 -0.79786927 -0.41168392]]] (2, 3, 8)
||dv/dtheta||: 2.4494895935058594 = sqrt(3*2)
df/dv^T * dv/dtheta:
[[-0.13219868 -0.13219868]
 [-0.17382757 -0.17382756]
 [ 0.7992858   0.7992858 ]

# Lemma O.6 from [EigenGame: PCA as a Nash Equilibrium](https://arxiv.org/abs/2010.00554)

In [ ]:
print('Testing Lemma O.6 from original EigenGame paper')

g = [- (eigs[i+1] - eigs[i]) for i in range(len(eigs)-1)]
print('g:', g)

c_i = 1/16
print('c_i:', c_i)

for attempt in range(10):
  print()
  print(f'Attempt {attempt}:')

  angle_dif = np.pi/4

  for i in range(1, k):
    rand_angle = randomize_angle(angle_dif)
    v_i = initialize_relative_vec(rand_angle, eigvecs[i-1])

    vecs = [initialize_relative_vec(randomize_angle(c_i*g[i-1]/(i*eigs[0])), eigvecs[j]) for j in range(i-1)]

    lhs = np.abs(rand_angle)
    rhs = np.linalg.norm(d_agent_utility(v_i, M, vecs)) * np.pi/g[i-1]

    print(f'i = {i}, {lhs} <= {rhs}')

Testing Lemma O.6 from original EigenGame paper
g: [0.43807676059894324, 0.02973724261513322, 0.014730934916612537, 0.17032758462547656, 0.011631726047975618, 0.009037646154562456, 0.012752419642939411]
c_i: 0.0625

Attempt 0:
i = 1, 0.4789926001653634 <= 7.527155370973013
i = 2, 0.6442386022532632 <= 70.5870596546693
i = 3, 0.11131185627772788 <= 247.9670776660586
i = 4, 0.3798124448677478 <= 15.522806471594754
i = 5, 0.699864476068343 <= 239.64896318875287
i = 6, 0.6869656874823326 <= 388.2050346410003
i = 7, 0.3981124466724541 <= 214.16374794311412

Attempt 1:
i = 1, 0.679185411904315 <= 5.723897705518619
i = 2, 0.20049588795032733 <= 108.15449808731027
i = 3, 0.22319069998598265 <= 228.07721159461263
i = 4, 0.6561344516095204 <= 13.146268097043073
i = 5, 0.2105576874958381 <= 304.82431558327295
i = 6, 0.491571955968862 <= 306.3826741177533
i = 7, 0.616922171140231 <= 199.7570964235377

Attempt 2:
i = 1, 0.48505912272625284 <= 7.393603747766897
i = 2, 0.0024285724407007736 <= 126.69

# Lemma 11

In [ ]:
print('Testing Lemma 11')

g = [- (eigs[i+1] - eigs[i]) for i in range(len(eigs)-1)]
print('g:', g)

c_i = 1/16
print('c_i:', c_i)

sigma = 1e-5

for attempt in range(10):
  print()
  print(f'Attempt {attempt}:')

  angle_dif = np.pi/4

  for i in range(1, k):
    rand_angle = randomize_angle(angle_dif)
    v_i = initialize_relative_vec(rand_angle, eigvecs[i-1])

    vecs = [initialize_relative_vec(randomize_angle(c_i*g[i-1]/(i*eigs[0])), eigvecs[j]) for j in range(i-1)]

    lhs = np.abs(rand_angle)
    rhs = np.linalg.norm(d_agent_utility_epsilon(v_i, M, vecs, epsilon=sigma)) * np.pi/g[i-1]

    print(f'i = {i}, {lhs} <= {rhs}')

Testing Lemma 11
g: [0.43807676059894324, 0.02973724261513322, 0.014730934916612537, 0.17032758462547656, 0.011631726047975618, 0.009037646154562456, 0.012752419642939411]
c_i: 0.0625

Attempt 0:
i = 1, 0.7849391345559291 <= 5.676207292089151
i = 2, 0.11167862921409441 <= 120.67794519402558
i = 3, 0.3944405829729537 <= 201.1163744764801
i = 4, 0.3589657633637133 <= 17.29617835524507
i = 5, 0.5921167480915165 <= 193.2090140442714
i = 6, 0.7050700241702915 <= 812.1957453296943
i = 7, 0.17847239808853957 <= 279.1614858305367

Attempt 1:
i = 1, 0.6370207370575965 <= 6.973407396663621
i = 2, 0.19047825729590329 <= 118.6032433081667
i = 3, 0.02747470111756263 <= 255.70827968197835
i = 4, 0.6860070046723309 <= 15.206357507101249
i = 5, 0.6010906502064965 <= 225.17213226063913
i = 6, 0.5581640764969861 <= 312.1949292009898
i = 7, 0.4858680836806949 <= 241.81443463927658

Attempt 2:
i = 1, 0.5050962547597413 <= 7.130833079090108
i = 2, 0.26403887742000753 <= 110.79029475412979
i = 3, 0.52928342

# Lemma O.8 from [EigenGame: PCA as a Nash Equilibrium](https://arxiv.org/abs/2010.00554)

In [ ]:
print('Testing Lemma O.8 from original EigenGame paper')

kappa = [eigs[0]/eigs[j] for j in range(1, len(eigs))]

for attempt in range(10):
  print()
  print(f'Attempt {attempt}:')

  for i in range(1, k):
    epsilon = randomize_angle(1)
    rand_angle = randomize_angle(np.pi/2)
    v_i = initialize_relative_vec(rand_angle, eigvecs[i-1])

    vecs = [initialize_relative_vec(randomize_angle(epsilon), eigvecs[j]) for j in range(i-1)]

    lhs = np.linalg.norm(d_agent_utility(v_i, M, vecs))
    rhs = 2*eigs[0] * (1 + (i-1)*(1+(1+kappa[i-1])*epsilon/(1-epsilon**2)**(1/2)))

    print(f'i = {i}, {lhs} <= {rhs}')

Testing Lemma O.8 from original EigenGame paper

Attempt 0:
i = 1, 0.9474900960922241 <= 1.5777095948201403
i = 2, 1.0067963600158691 <= 6.841195804120294
i = 3, 1.11008620262146 <= 7.792118153377506
i = 4, 0.8042078614234924 <= 10.482624067913445
i = 5, 7.365488052368164 <= 38.95820821941593
i = 6, 4.8493757247924805 <= 11.651813488180734
i = 7, 2.091048002243042 <= 20.43076505423994

Attempt 1:
i = 1, 1.0855547189712524 <= 1.5777095948201403
i = 2, 0.9635361433029175 <= 4.765353234252093
i = 3, 0.6242443919181824 <= 30.771865931207998
i = 4, 3.9829928874969482 <= 56.22459077209545
i = 5, 1.1203824281692505 <= 52.32925502189512
i = 6, 2.047588348388672 <= 9.948895405833376
i = 7, 2.0454325675964355 <= 74.81260415005485

Attempt 2:
i = 1, 0.19153350591659546 <= 1.5777095948201403
i = 2, 0.8377701640129089 <= 5.014193782399188
i = 3, 0.4479050040245056 <= 22.98256379440525
i = 4, 1.0201798677444458 <= 25.56893990459866
i = 5, 0.7963667511940002 <= 30.64230066436812
i = 6, 1.082209706306

# Lemma 6

In [ ]:
print('Testing Lemma 6')

kappa = [eigs[0]/eigs[j] for j in range(1, len(eigs))]

sigma = 1e-5

for attempt in range(10):
  print()
  print(f'Attempt {attempt}:')

  for i in range(1, k):
    epsilon = randomize_angle(1)
    rand_angle = randomize_angle(np.pi/2)
    v_i = initialize_relative_vec(rand_angle, eigvecs[i-1])

    vecs = [initialize_relative_vec(randomize_angle(epsilon), eigvecs[j]) for j in range(i-1)]

    lhs = np.linalg.norm(d_agent_utility_epsilon(v_i, M, vecs, epsilon=1e-5))
    rhs = 2 * eigs[0] * (1 + (i-1) * (1+(1+kappa[i-1])*epsilon/(1-epsilon**2)**(1/2))) + sigma * (np.linalg.norm(M) + i * ( eigs[0] * kappa[i-1] / (1-epsilon**2)))

    print(f'i = {i}, {lhs} <= {rhs}')

Testing Lemma 6

Attempt 0:
i = 1, 1.1141287088394165 <= 1.5777374492392153
i = 2, 1.2086889743804932 <= 17.18944383600991
i = 3, 0.3990589380264282 <= 14.291782202200931
i = 4, 1.172015905380249 <= 13.104817569711866
i = 5, 3.7005505561828613 <= 31.4514743742831
i = 6, 0.4373609721660614 <= 259.7496108325987
i = 7, 1.0460572242736816 <= 16.87536058214159

Attempt 1:
i = 1, 0.4120500981807709 <= 1.5777464344692491
i = 2, 0.41052430868148804 <= 11.825121457876865
i = 3, 2.464647054672241 <= 8.365062510216525
i = 4, 1.190049648284912 <= 81.21298506794517
i = 5, 3.283431053161621 <= 23.80720519502665
i = 6, 1.1841989755630493 <= 129.58558799201052
i = 7, 2.1690850257873535 <= 36.847997337276524

Attempt 2:
i = 1, 0.49474549293518066 <= 1.577737597088488
i = 2, 0.32581448554992676 <= 7.648841760687318
i = 3, 0.5941360592842102 <= 14.83234826710268
i = 4, 4.400712490081787 <= 29.35787279395499
i = 5, 1.2692850828170776 <= 60.26739261629171
i = 6, 1.0544464588165283 <= 19.538577790574394
i =

# Lemma O.9 from [EigenGame: PCA as a Nash Equilibrium](https://arxiv.org/abs/2010.00554)

In [ ]:
print('Testing Lemma O.9 from original EigenGame paper')
print()

g = [- (eigs[i+1] - eigs[i]) for i in range(len(eigs)-1)]
print('g:', g)

c_i = 1/16
print('c_i:', c_i)

kappa = [eigs[0]/eigs[j] for j in range(1, len(eigs))]

for attempt in range(10):
  print()
  print(f'Attempt {attempt}:')

  for i in range(1, k):
    epsilon = randomize_angle(c_i*g[i-1]/(i*eigs[0]))
    rand_angle = randomize_angle(np.pi/2)
    v_i = initialize_relative_vec(rand_angle, eigvecs[i-1])

    vecs = [initialize_relative_vec(randomize_angle(epsilon), eigvecs[j]) for j in range(i-1)]

    lhs = np.linalg.norm(d_agent_utility(v_i, M, vecs))
    rhs = 4*(eigs[0]*i + (1+kappa[i-1])*c_i*g[i-1])

    print(f'i = {i}, {lhs} <= {rhs}')

Testing Lemma O.9 from original EigenGame paper

g: [0.43807676059894324, 0.02973724261513322, 0.014730934916612537, 0.17032758462547656, 0.011631726047975618, 0.009037646154562456, 0.012752419642939411]
c_i: 0.0625

Attempt 0:
i = 1, 0.6696212291717529 <= 3.5112329863266374
i = 2, 1.0188276767730713 <= 6.336540124262623
i = 3, 1.1128044128417969 <= 9.479424627368308
i = 4, 4.81241512298584 <= 12.911283005143039
i = 5, 1.2537750005722046 <= 15.79845121066202
i = 6, 0.9224324822425842 <= 18.95023116866308
i = 7, 0.9572177529335022 <= 22.115644078397207

Attempt 1:
i = 1, 0.2114339917898178 <= 3.5112329863266374
i = 2, 1.0521711111068726 <= 6.336540124262623
i = 3, 2.1344645023345947 <= 9.479424627368308
i = 4, 0.8834030628204346 <= 12.911283005143039
i = 5, 3.7946653366088867 <= 15.79845121066202
i = 6, 0.8383870124816895 <= 18.95023116866308
i = 7, 0.9153802990913391 <= 22.115644078397207

Attempt 2:
i = 1, 0.5371983051300049 <= 3.5112329863266374
i = 2, 0.8406451344490051 <= 6.3365401

# Lemma 7

In [ ]:
print('Testing Lemma 7')
print()

g = [- (eigs[i+1] - eigs[i]) for i in range(len(eigs)-1)]
print('g:', g)

c_i = 1/16
print('c_i:', c_i)

kappa = [eigs[0]/eigs[j] for j in range(1, len(eigs))]

sigma = 1e-5

for attempt in range(10):
  print()
  print(f'Attempt {attempt}:')

  for i in range(1, k):
    epsilon = randomize_angle(c_i*g[i-1]/(i*eigs[0]))
    rand_angle = randomize_angle(np.pi/2)
    v_i = initialize_relative_vec(rand_angle, eigvecs[i-1])

    vecs = [initialize_relative_vec(randomize_angle(epsilon), eigvecs[j]) for j in range(i-1)]

    lhs = np.linalg.norm(d_agent_utility_epsilon(v_i, M, vecs, epsilon=sigma))
    rhs = 4*(eigs[0]*i + (1+kappa[i-1])*c_i*g[i-1]) + sigma*(np.linalg.norm(np.diag(M)) + 2*(i-1)*eigs[0]*kappa[i-1])

    print(f'i = {i}, {lhs} <= {rhs}')

Testing Lemma 7

g: [0.43807676059894324, 0.02973724261513322, 0.014730934916612537, 0.17032758462547656, 0.011631726047975618, 0.009037646154562456, 0.012752419642939411]
c_i: 0.0625

Attempt 0:
i = 1, 0.3850984573364258 <= 3.511242878321623
i = 2, 1.0730615854263306 <= 6.336588783411356
i = 3, 1.9160089492797852 <= 9.479515782420625
i = 4, 4.268254280090332 <= 12.911567473464821
i = 5, 1.13173508644104 <= 15.798861449514094
i = 6, 4.274173736572266 <= 18.95078071569941
i = 7, 5.805541515350342 <= 22.116382077601937

Attempt 1:
i = 1, 1.1916457414627075 <= 3.511242878321623
i = 2, 1.0766170024871826 <= 6.336588783411356
i = 3, 0.8219608068466187 <= 9.479515782420625
i = 4, 4.24843168258667 <= 12.911567473464821
i = 5, 4.508538722991943 <= 15.798861449514094
i = 6, 7.009082794189453 <= 18.95078071569941
i = 7, 0.9461944103240967 <= 22.116382077601937

Attempt 2:
i = 1, 1.1988834142684937 <= 3.511242878321623
i = 2, 0.6133995652198792 <= 6.336588783411356
i = 3, 1.1389790773391724 <= 9.